# I\. Setup and load dataframes

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.float_format", "{:.2f}".format)


In [2]:
# Load the dataframes
albums_df = pd.read_csv("./pipeline/4.5.Albums_join_everything.csv")
artists_df = pd.read_csv("./pipeline/4.5.Artists_join_everything.csv")
tracks_df = pd.read_csv("./pipeline/4.5.Tracks_join_everything.csv")
wide_df = pd.read_csv("./pipeline/4.5.Wide_join_everything.csv")

/tmp/ipykernel_913/1055989077.py:5: DtypeWarning: Columns (45,49,58,84) have mixed types. Specify dtype option on import or set low_memory=False.
  wide_df = pd.read_csv("./pipeline/4.5.Wide_join_everything.csv")


# I\.2 Cross\-entity helper functions

In [3]:
def drop_missing_or_nonpositive(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """
    Drop rows where `col` is NaN or <= 0.
    Used for any Last.fm count metric we want to gate on.
    """
    return df.loc[df[col].notna() & (df[col] > 0)].copy()


def safe_log(series: pd.Series) -> pd.Series:
    """
    Log-transform safely:
      - log(x) when x > 0
      - NaN otherwise
    """
    return np.where(series > 0, np.log(series), np.nan)


def add_log_cols(df: pd.DataFrame, cols: list[str], prefix: str = "log_") -> pd.DataFrame:
    """
    Add log-transformed columns for each metric in `cols`.
    Creates new columns named f"{prefix}{col}".
    """
    out = df.copy()
    for c in cols:
        out[f"{prefix}{c}"] = safe_log(out[c])
    return out


# II\. Transform and reduce albums

Let's tackle albums first\.\.\.

In [4]:
# =============================================================================
# II. Transform and Reduce to Analytics Set (ALBUMS)
# -----------------------------------------------------------------------------
# Goal:
#   Create albums_analytics_df: a clean “analytics-ready” subset of albums_df
#   that:
#     1) Keeps only canonical soundtrack albums for films with sufficient vote_count
#     2) Removes rows where Last.fm album listeners are missing or zero
#     3) Adds log-transformed Last.fm album metrics for analysis
#
# Why we do this:
#   - Canonical soundtrack filter ensures a stable 1:1-ish film→album representation
#     (based on our canonical rules), reducing duplication and ambiguity.
#   - vote_count_above_500 is a quality gate so we focus on films with enough
#     TMDB activity to be meaningful in downstream analyses.
#   - Dropping 0/NaN listeners removes “no signal” rows that can distort
#     distributional analysis and break log transforms.
#   - Log transforms help stabilize heavy-tailed popularity distributions
#     (common with playcounts/listeners).
# =============================================================================


# -----------------------------------------------------------------------------
# Album-specific helper functions
# -----------------------------------------------------------------------------
def filter_to_canonical_high_confidence_albums(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter to the albums we consider "analytics eligible" for this milestone:
      - is_canonical_soundtrack == True
      - vote_count_above_500 == True

    Assumptions:
      - The incoming dataframe already has these boolean columns.
      - We do NOT deduplicate or rename columns here (keeps the notebook readable).
    """
    mask = (df["is_canonical_soundtrack"] == True) & (df["vote_count_above_500"] == True)
    return df.loc[mask].copy()


QA block\. This helper prints a quick, standardized QA snapshot for the album\-level transformation pipeline\. It makes the filtering steps auditable by showing row\-count attrition \(baseline → canonical/vote filter → listener\-clean\), quantifying how many missing/zero listener values were removed, and sanity\-checking the remaining listener distribution \(min/median/max\)

In [5]:
def qa_print_album_transform_stats(
    df_before: pd.DataFrame,
    df_after_canonical: pd.DataFrame,
    df_after_listener_clean: pd.DataFrame,
    listeners_col: str = "lfm_album_listeners"
) -> None:
    """
    Print lightweight QA stats so teammates can sanity-check the transformation.
    Keeps the notebook transparent and easy to audit.
    """
    print("ALBUM ANALYTICS TRANSFORM — QA SUMMARY")
    print("-" * 60)

    # Row counts
    print(f"Rows before:                          {len(df_before):,}")
    print(f"After canonical + vote_count filter:  {len(df_after_canonical):,}")
    print(f"After drop missing/0 listeners:       {len(df_after_listener_clean):,}")

    # Listener completeness + range
    missing_listeners = df_after_canonical[listeners_col].isna().sum()
    zero_listeners = (df_after_canonical[listeners_col] == 0).sum()

    print("-" * 60)
    print(f"Listener NaNs removed (from filtered): {missing_listeners:,}")
    print(f"Listener zeros removed (from filtered): {zero_listeners:,}")

    if len(df_after_listener_clean) > 0:
        print("-" * 60)
        print("Post-clean listener sanity checks:")
        print(f"  min listeners: {df_after_listener_clean[listeners_col].min():,}")
        print(f"  median:        {df_after_listener_clean[listeners_col].median():,}")
        print(f"  max listeners: {df_after_listener_clean[listeners_col].max():,}")
    print("-" * 60)
    print()

In [6]:
# -----------------------------------------------------------------------------
# Build albums_analytics_df
# -----------------------------------------------------------------------------
# Step 0: keep the raw df as the “before” reference
albums_before_df = albums_df.copy()

# Step 1: Filter to canonical soundtrack albums where TMDB vote_count is strong enough
albums_filtered_df = filter_to_canonical_high_confidence_albums(albums_before_df)

# Step 2: Drop rows with no usable Last.fm listener signal (0 or NaN)
albums_clean_df = drop_missing_or_nonpositive(albums_filtered_df, "lfm_album_listeners")

# Step 3: Add log transforms for album listeners + playcount
albums_analytics_df = add_log_cols(
    albums_clean_df,
    cols=["lfm_album_listeners", "lfm_album_playcount"]
)

# QA printout (optional but recommended in the notebook)
qa_print_album_transform_stats(
    df_before=albums_before_df,
    df_after_canonical=albums_filtered_df,
    df_after_listener_clean=albums_analytics_df,
    listeners_col="lfm_album_listeners"
)

# At this point, albums_analytics_df is ready for downstream analysis.
# Let's take a quick peek:
display(albums_analytics_df.head(3).round(2))
display(albums_analytics_df[["lfm_album_listeners","log_lfm_album_listeners","lfm_album_playcount","log_lfm_album_playcount"]].describe().round(2))


ALBUM ANALYTICS TRANSFORM — QA SUMMARY
------------------------------------------------------------
Rows before:                          4,771
After canonical + vote_count filter:  1,622
After drop missing/0 listeners:       1,551
------------------------------------------------------------
Listener NaNs removed (from filtered): 13
Listener zeros removed (from filtered): 58
------------------------------------------------------------
Post-clean listener sanity checks:
  min listeners: 1.0
  median:        27.0
  max listeners: 1,034,714.0
------------------------------------------------------------



,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,bafta_score_nominee,bafta_score_winner,lfm_album_status,lfm_album_listeners,lfm_album_playcount,lfm_album_url,lfm_album_query_method,lfm_album_pulled_at,log_lfm_album_listeners,log_lfm_album_playcount
1,216015,Fifty Shades of Grey,False,125,"Drama, Romance, Thriller",5.88,12222,R,Fifty Shades of Grey,English,...,False,False,ok,27.00,420.00,https://www.last.fm/music/Danny+Elfman/Fifty+S...,text,2026-01-28 22:18:03.783559+00:00,3.30,6.04
2,150689,Cinderella,False,105,"Romance, Fantasy, Family, Drama",6.83,7294,PG,Cinderella,English,...,False,False,ok,524.00,16416.00,https://www.last.fm/music/Patrick+Doyle/Cinder...,text,2026-01-28 21:37:16.320393+00:00,6.26,9.71
3,281957,The Revenant,False,157,"Western, Drama, Adventure",7.54,18928,R,The Revenant,English,...,True,True,ok,14.00,148.00,https://www.last.fm/music/%E5%9D%82%E6%9C%AC%E...,text,2026-01-28 21:42:42.448069+00:00,2.64,5.00


,lfm_album_listeners,log_lfm_album_listeners,lfm_album_playcount,log_lfm_album_playcount
count,1551.00,1551.00,1551.00,1551.00
mean,3800.20,3.91,48582.95,6.24
std,35094.87,2.65,542295.85,2.94
min,1.00,0.00,1.00,0.00
25%,8.00,2.08,68.00,4.22
50%,27.00,3.30,404.00,6.00
75%,246.00,5.51,3418.50,8.14
max,1034714.00,13.85,18282803.00,16.72


Findings: After applying canonical soundtrack and vote\-count quality gates, then removing albums with no listener signal, the final albums analytics set contains 1,551 soundtrack albums with valid Last\.fm engagement\. As expected, both listener and playcount distributions are highly right\-skewed, with a small number of breakout soundtracks driving the upper tail\. The log transforms substantially compress this skew, yielding more stable, analysis\-ready variables that better support correlation analysis, modeling, and cross\-film comparisons in downstream sections\.

# III\. Transform and reduce artists

In [7]:
# =============================================================================
# III. Transform and Reduce to Analytics Set (ARTISTS)
# -----------------------------------------------------------------------------
# Goal:
#   Create artists_analytics_df: an analytics-ready subset of artists_df that:
#     1) Removes artists with no usable Last.fm listener signal (NaN or 0)
#     2) Adds log-transformed Last.fm artist popularity metrics
#
# Why we do this (vs the album approach):
#   - Artists are already a derived "spine universe" (artists appearing on
#     soundtrack release groups). Applying film-level gates (canonical soundtrack,
#     vote_count thresholds) would arbitrarily exclude artists and conflate
#     artist popularity with film popularity.
#   - We gate on listeners because it is the cleanest indicator that Last.fm
#     returned meaningful engagement data for that artist.
# =============================================================================

def qa_print_artist_transform_stats(
    df_before: pd.DataFrame,
    df_after_listener_clean: pd.DataFrame,
    listeners_col: str = "lfm_artist_listeners"
) -> None:
    """
    Print lightweight QA stats so teammates can sanity-check the transformation.
    """
    print("ARTIST ANALYTICS TRANSFORM — QA SUMMARY")
    print("-" * 60)

    print(f"Rows before:                     {len(df_before):,}")
    print(f"After drop missing/0 listeners:  {len(df_after_listener_clean):,}")

    missing_listeners = df_before[listeners_col].isna().sum()
    zero_listeners = (df_before[listeners_col] == 0).sum()

    print("-" * 60)
    print(f"Listener NaNs removed: {missing_listeners:,}")
    print(f"Listener zeros removed: {zero_listeners:,}")

    if len(df_after_listener_clean) > 0:
        print("-" * 60)
        print("Post-clean listener sanity checks:")
        print(f"  min listeners: {df_after_listener_clean[listeners_col].min():,}")
        print(f"  median:        {df_after_listener_clean[listeners_col].median():,}")
        print(f"  max listeners: {df_after_listener_clean[listeners_col].max():,}")
    print("-" * 60)
    print()


In [8]:
# -----------------------------------------------------------------------------
# Build artists_analytics_df
# -----------------------------------------------------------------------------
# Step 0: keep the raw df as the “before” reference
artists_before_df = artists_df.copy()

# Step 1: Drop rows with no usable Last.fm listener signal (0 or NaN)
artists_clean_df = drop_missing_or_nonpositive(artists_before_df, "lfm_artist_listeners")

# Step 2: Add log transforms for artist listeners + playcount
artists_analytics_df = add_log_cols(
    artists_clean_df,
    cols=["lfm_artist_listeners", "lfm_artist_playcount"]
)

# QA printout (optional: recommended only in the notebook)
qa_print_artist_transform_stats(
    df_before=artists_before_df,
    df_after_listener_clean=artists_analytics_df,
    listeners_col="lfm_artist_listeners"
)


display(artists_analytics_df.head(3).round(2))
display(artists_analytics_df[["lfm_artist_listeners","log_lfm_artist_listeners",
                             "lfm_artist_playcount","log_lfm_artist_playcount"]].describe().round(2))


ARTIST ANALYTICS TRANSFORM — QA SUMMARY
------------------------------------------------------------
Rows before:                     2,430
After drop missing/0 listeners:  2,378
------------------------------------------------------------
Listener NaNs removed: 52
Listener zeros removed: 0
------------------------------------------------------------
Post-clean listener sanity checks:
  min listeners: 1.0
  median:        7,657.0
  max listeners: 7,572,615.0
------------------------------------------------------------



,artist_id,artist_mbid,name,sort_name,artist_type,gender,artist_comment,begin_date_year,begin_date_month,begin_date_day,...,bafta_score_nominee,bafta_score_winner,lfm_artist_status,lfm_artist_listeners,lfm_artist_playcount,lfm_artist_url,lfm_artist_query_method,lfm_artist_pulled_at,log_lfm_artist_listeners,log_lfm_artist_playcount
0,15,974fa09f-8761-4fbb-b156-390417d125ae,Éric Serra,"Serra, Éric",Person,Male,French bassist and score composer,1959.00,9.00,9.00,...,False,False,ok,84889.00,1138278.00,https://www.last.fm/music/%C3%89ric+Serra,mbid,2026-01-28 22:27:30.725322+00:00,11.35,13.95
1,25,36bfa85f-737b-41db-a8fc-b8825850ffc3,Pavement,Pavement,Group,NaN,US indie rock band,1989.00,NaN,NaN,...,False,False,ok,2127077.00,99575467.00,https://www.last.fm/music/Pavement,mbid,2026-01-28 22:27:30.725322+00:00,14.57,18.42
2,35,05755bf1-380c-487f-983f-d1a02401fa28,Cat Power,Cat Power,Person,Female,NaN,1972.00,1.00,21.00,...,False,False,ok,2229724.00,88009686.00,https://www.last.fm/music/Cat+Power,mbid,2026-01-28 22:27:30.725322+00:00,14.62,18.29


,lfm_artist_listeners,log_lfm_artist_listeners,lfm_artist_playcount,log_lfm_artist_playcount
count,2378.00,2378.00,2378.00,2378.00
mean,193315.60,8.69,9311976.14,11.04
std,618000.27,3.39,63679073.60,3.90
min,1.00,0.00,1.00,0.00
25%,628.50,6.44,4829.75,8.48
50%,7657.00,8.94,74125.50,11.21
75%,75994.50,11.24,1038632.25,13.85
max,7572615.00,15.84,1519682694.00,21.14


Findings: The artist analytics set includes 2,378 artists with valid Last\.fm engagement after removing records with missing listener data\. Artist popularity exhibits an even more pronounced long\-tail distribution than albums, with median listener counts \(~7\.7K\) far below the mean \(~193K\), reflecting a small number of globally prominent artists alongside a large base of moderately known composers and performers\. Log\-transforming listener and playcount metrics meaningfully compresses this skew, producing more stable representations of artist popularity that are better suited for comparative analysis across roles, genres, and soundtrack participation\.

# IV\. Transform and reduce tracks

Let's now cleanup the tracks table and reduce it to the analytics set

In [9]:
# =============================================================================
# IV. Transform and Reduce to Analytics Set (TRACKS)
# -----------------------------------------------------------------------------
# Goal:
#   Create tracks_analytics_df containing:
#     - Tracks belonging to a high-confidence soundtrack album universe
#       (canonical soundtrack + vote_count_above_500)
#     - Tracks with valid Last.fm playcount signal
#     - Log-transformed playcount for analysis
#
# Rationale:
#   - Tracks are analyzed independently from albums in terms of popularity.
#   - However, tracks must belong to albums that pass our soundtrack quality
#     gates to ensure a valid film→soundtrack context.
#   - Playcount is the primary engagement metric at the track level.
# =============================================================================


# -----------------------------------------------------------------------------
# Helper functions
# -----------------------------------------------------------------------------
def filter_tracks_to_album_universe(
    tracks_df: pd.DataFrame,
    albums_df: pd.DataFrame,
    album_key: str = "release_group_id"
) -> pd.DataFrame:
    """
    Restrict tracks to those belonging to albums that passed the
    canonical soundtrack + vote_count quality gate.

    Assumptions:
      - albums_df has already been filtered to canonical + vote_count_above_500.
      - album_key exists in both dataframes.
    """
    valid_album_ids = albums_df[album_key].unique()
    return tracks_df.loc[tracks_df[album_key].isin(valid_album_ids)].copy()



In [10]:
def qa_print_track_transform_stats(
    df_before: pd.DataFrame,
    df_after_album_filter: pd.DataFrame,
    df_after_playcount_clean: pd.DataFrame,
    playcount_col: str = "lfm_track_playcount"
) -> None:
    """
    Print lightweight QA stats for the track transformation.
    """
    print("TRACK ANALYTICS TRANSFORM — QA SUMMARY")
    print("-" * 60)

    print(f"Rows before:                         {len(df_before):,}")
    print(f"After album universe filter:         {len(df_after_album_filter):,}")
    print(f"After drop missing/0 playcounts:     {len(df_after_playcount_clean):,}")

    missing_pc = df_after_album_filter[playcount_col].isna().sum()
    zero_pc = (df_after_album_filter[playcount_col] == 0).sum()

    print("-" * 60)
    print(f"Playcount NaNs removed: {missing_pc:,}")
    print(f"Playcount zeros removed: {zero_pc:,}")

    if len(df_after_playcount_clean) > 0:
        print("-" * 60)
        print("Post-clean playcount sanity checks:")
        print(f"  min playcount: {df_after_playcount_clean[playcount_col].min():,}")
        print(f"  median:        {df_after_playcount_clean[playcount_col].median():,}")
        print(f"  max playcount: {df_after_playcount_clean[playcount_col].max():,}")
    print("-" * 60)
    print()

In [11]:
# -----------------------------------------------------------------------------
# Build tracks_analytics_df
# -----------------------------------------------------------------------------
tracks_before_df = tracks_df.copy()

# Step 1: Restrict to the canonical soundtrack album universe
tracks_album_filtered_df = filter_tracks_to_album_universe(
    tracks_before_df,
    albums_df=albums_filtered_df,  # from the album section (canonical + vote_count)
    album_key="release_group_id"
)

# Step 2: Drop tracks with no usable playcount signal
tracks_clean_df = drop_missing_or_nonpositive(tracks_album_filtered_df, "lfm_track_playcount")

# Step 3: Add log-transformed playcount
tracks_analytics_df = add_log_cols(
    tracks_clean_df,
    cols=["lfm_track_playcount"]
)

# QA printout
qa_print_track_transform_stats(
    df_before=tracks_before_df,
    df_after_album_filter=tracks_album_filtered_df,
    df_after_playcount_clean=tracks_analytics_df,
    playcount_col="lfm_track_playcount"
)

display(tracks_analytics_df.head(3).round(2))
display(tracks_analytics_df[["lfm_track_playcount","log_lfm_track_playcount"]].describe().round(2))


TRACK ANALYTICS TRANSFORM — QA SUMMARY
------------------------------------------------------------
Rows before:                         78,992
After album universe filter:         33,878
After drop missing/0 playcounts:     32,261
------------------------------------------------------------
Playcount NaNs removed: 1,617
Playcount zeros removed: 0
------------------------------------------------------------
Post-clean playcount sanity checks:
  min playcount: 1.0
  median:        463.0
  max playcount: 31,681,683.0
------------------------------------------------------------



,tmdb_id,film_title,release_group_id,release_id,match_method,album_us_release_date,us_date_has_missing_month,us_date_has_missing_day,medium_id,disc_number,...,popularity,energy,danceability,happiness,acousticness,instrumentalness,liveness,speechiness,loudness,log_lfm_track_playcount
0,158852,Tomorrowland,1527294,1608979,imdb_exact,2015-06-09,False,False,1692109,1,...,41.00,55.00,84.00,63.00,7.00,0.00,17.00,30.00,-9 dB,7.23
2,524434,Eternals,2804779,3233921,imdb_exact,NaN,NaN,NaN,3523483,1,...,63.00,99.00,31.00,4.00,0.00,9.00,10.00,32.00,-4 dB,8.99
7,302946,The Accountant,1765530,1914190,title_contains_strict,2016-10-01,False,False,2044240,1,...,15.00,9.00,24.00,8.00,99.00,91.00,14.00,4.00,-26 dB,8.10


,lfm_track_playcount,log_lfm_track_playcount
count,32261.00,32261.00
mean,65054.52,6.31
std,735602.09,2.72
min,1.00,0.00
25%,101.00,4.62
50%,463.00,6.14
75%,2624.00,7.87
max,31681683.00,17.27


Findings: The track analytics set contains 32,261 tracks with valid Last\.fm playcount, all belonging to canonical soundtrack albums for films that passed the vote\-count quality gate\. Track engagement is strongly right\-skewed: while the median track has only a few hundred plays, a small number of standout tracks reach into the tens of millions\. Applying a log transform substantially compresses this range, producing a more stable representation of track\-level engagement that supports comparative analysis across films, albums, and soundtrack components in subsequent sections\.

# V\. Transform and reduce wide

Finally, let's handle the wide table

In [12]:
# =============================================================================
# V. Transform and Reduce to Analytics Set (WIDE)
# -----------------------------------------------------------------------------
# Goal:
#   Create wide_analytics_df at track-grain, aligned to the exact same
#   universe as tracks_analytics_df.
#
# Universe definition (by design):
#   - Belongs to the canonical soundtrack + vote_count_above_500 album universe
#   - Has valid Last.fm track playcount signal
#
# Notes:
#   - We intentionally DO NOT filter wide rows based on artist-level Last.fm
#     completeness. Artist metrics are treated as enrichment and may be null.
#   - We DO log-transform all Last.fm numeric metrics in wide (track, album,
#     and artist), using a safe log to avoid -inf when values are missing/0.
# =============================================================================


# -----------------------------------------------------------------------------
# Helper functions
# -----------------------------------------------------------------------------
def filter_wide_to_track_universe(wide_df: pd.DataFrame, tracks_analytics_df: pd.DataFrame) -> pd.DataFrame:
    """
    Restrict wide_df to the track universe already defined by tracks_analytics_df.

    This is the cleanest way to guarantee that 'tracks_analytics_df' and
    'wide_analytics_df' represent the exact same set of track rows.
    """
    valid_track_ids = tracks_analytics_df["track_id"].unique()
    return wide_df.loc[wide_df["track_id"].isin(valid_track_ids)].copy()


In [13]:
def qa_print_wide_transform_stats(
    df_before: pd.DataFrame,
    df_after_track_filter: pd.DataFrame
) -> None:
    """
    Lightweight QA stats for wide transformation.
    """
    print("WIDE ANALYTICS TRANSFORM — QA SUMMARY")
    print("-" * 60)
    print(f"Rows before:                    {len(df_before):,}")
    print(f"After track universe filter:    {len(df_after_track_filter):,}")

    # Quick sanity checks for track playcount (the metric that defines the universe)
    if len(df_after_track_filter) > 0:
        print("-" * 60)
        print("Post-filter sanity checks (track playcount):")
        print(f"  track playcount min: {df_after_track_filter['lfm_track_playcount'].min():,}")
        print(f"  track playcount med: {df_after_track_filter['lfm_track_playcount'].median():,}")
        print(f"  track playcount max: {df_after_track_filter['lfm_track_playcount'].max():,}")

        # Enrichment completeness (informational only)
        album_listener_nulls = df_after_track_filter["lfm_album_listeners"].isna().sum()
        artist_listener_nulls = df_after_track_filter["lfm_album_primary_artist_listeners"].isna().sum()

        print("-" * 60)
        print("Enrichment completeness (not used for filtering):")
        print(f"  lfm_album_listeners nulls:                 {album_listener_nulls:,}")
        print(f"  lfm_album_primary_artist_listeners nulls:  {artist_listener_nulls:,}")

    print("-" * 60)
    print()


In [14]:
# -----------------------------------------------------------------------------
# Build wide_analytics_df
# -----------------------------------------------------------------------------
wide_before_df = wide_df.copy()

# Step 1: Filter to the same track universe as tracks_analytics_df
wide_filtered_df = filter_wide_to_track_universe(wide_before_df, tracks_analytics_df)

# Step 2: Add log transforms for all Last.fm metrics in wide
wide_analytics_df = add_log_cols(
    wide_filtered_df,
    cols=[
        "lfm_track_playcount", "lfm_track_listeners",
        "lfm_album_playcount", "lfm_album_listeners",
        "lfm_album_primary_artist_playcount", "lfm_album_primary_artist_listeners",
    ]
)

# QA printout
qa_print_wide_transform_stats(
    df_before=wide_before_df,
    df_after_track_filter=wide_analytics_df
)

display(wide_analytics_df.head(3).round(2))

display(
    wide_analytics_df[
        [
            "lfm_track_playcount", "log_lfm_track_playcount",
            "lfm_track_listeners", "log_lfm_track_listeners",
            "lfm_album_playcount", "log_lfm_album_playcount",
            "lfm_album_listeners", "log_lfm_album_listeners",
            "lfm_album_primary_artist_playcount", "log_lfm_album_primary_artist_playcount",
            "lfm_album_primary_artist_listeners", "log_lfm_album_primary_artist_listeners",
        ]
    ].describe().round(2)
)

WIDE ANALYTICS TRANSFORM — QA SUMMARY
------------------------------------------------------------
Rows before:                    78,992
After track universe filter:    32,261
------------------------------------------------------------
Post-filter sanity checks (track playcount):
  track playcount min: 1.0
  track playcount med: 463.0
  track playcount max: 31,681,683.0
------------------------------------------------------------
Enrichment completeness (not used for filtering):
  lfm_album_listeners nulls:                 109
  lfm_album_primary_artist_listeners nulls:  17
------------------------------------------------------------

/root/venv/lib/python3.11/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/root/venv/lib/python3.11/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,instrumentalness,liveness,speechiness,loudness,log_lfm_track_playcount,log_lfm_track_listeners,log_lfm_album_playcount,log_lfm_album_listeners,log_lfm_album_primary_artist_playcount,log_lfm_album_primary_artist_listeners
240,407445,Breathe,False,118,"Drama, Romance",7.40,785,PG-13,Breathe,English,...,90.00,7.00,3.00,-23 dB,5.92,3.26,8.26,6.73,15.15,12.89
241,407445,Breathe,False,118,"Drama, Romance",7.40,785,PG-13,Breathe,English,...,93.00,15.00,4.00,-29 dB,6.10,3.47,8.26,6.73,15.15,12.89
242,407445,Breathe,False,118,"Drama, Romance",7.40,785,PG-13,Breathe,English,...,89.00,8.00,5.00,-32 dB,5.91,3.47,8.26,6.73,15.15,12.89


,lfm_track_playcount,log_lfm_track_playcount,lfm_track_listeners,log_lfm_track_listeners,lfm_album_playcount,log_lfm_album_playcount,lfm_album_listeners,log_lfm_album_listeners,lfm_album_primary_artist_playcount,log_lfm_album_primary_artist_playcount,lfm_album_primary_artist_listeners,log_lfm_album_primary_artist_listeners
count,32261.00,32261.00,32261.00,32261.00,32152.00,31265.00,32152.00,31265.00,32244.00,32244.00,32244.00,32244.00
mean,65054.52,6.31,9881.28,5.37,53325.18,6.35,3932.77,3.93,11378323.00,14.32,383874.15,11.62
std,735602.09,2.72,84833.40,2.44,644474.68,2.87,39455.18,2.61,44934481.66,2.40,605453.27,2.12
min,1.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00
25%,101.00,4.62,48.00,3.87,67.00,4.38,7.00,2.08,415669.00,12.94,37718.00,10.54
50%,463.00,6.14,182.00,5.20,395.00,6.05,25.00,3.33,2870337.00,14.87,198541.00,12.20
75%,2624.00,7.87,868.00,6.77,3365.00,8.16,200.00,5.41,9561884.00,16.07,468934.00,13.06
max,31681683.00,17.27,3045671.00,14.93,18282803.00,16.72,1034714.00,13.85,996998103.00,20.72,7572615.00,15.84


Findings:  The wide analytics set contains 32,261 track\-level rows, aligned exactly to the validated track universe derived from canonical soundtrack albums and films that passed the vote\-count quality gate\. Track playcount in the wide table mirrors the track\-only analytics set, with a pronounced right skew driven by a small number of highly popular tracks\. 

To make these heavy\-tailed engagement signals more analysis\-ready, we log\-transform all available Last\.fm metrics in the wide table \(track, album, and album\-primary\-artist listeners/playcounts\)\. This stabilizes distributions for downstream comparisons while preserving the full film, album, and artist context—and it does so without imposing additional filtering on enrichment fields, which may be null when upstream Last\.fm data is missing\.

# VI\. Write to file

In [15]:
# =============================================================================
# Persist Analytics Sets to Disk
# -----------------------------------------------------------------------------
# We write each analytics-ready dataframe to disk so downstream analysis
# notebooks can load a stable, frozen version of the 4.7 outputs.
#
# Files written:
#   ./pipeline/4.7.albums_analytics_set.csv
#   ./pipeline/4.7.artists_analytics_set.csv
#   ./pipeline/4.7.tracks_analytics_set.csv
#   ./pipeline/4.7.wide_analytics_set.csv
# =============================================================================

OUTPUT_DIR = "./pipeline"

albums_analytics_df.to_csv(
    f"{OUTPUT_DIR}/4.7.Albums_analytics_set.csv",
    index=False
)

artists_analytics_df.to_csv(
    f"{OUTPUT_DIR}/4.7.Artists_analytics_set.csv",
    index=False
)

tracks_analytics_df.to_csv(
    f"{OUTPUT_DIR}/4.7.Tracks_analytics_set.csv",
    index=False
)

wide_analytics_df.to_csv(
    f"{OUTPUT_DIR}/4.7.Wide_analytics_set.csv",
    index=False
)

print("4.7 analytics sets successfully written to disk.")


4.7 analytics sets successfully written to disk.


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b124131d-2024-4253-bb46-8043aed4b78f' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>